# 9장. 회귀 분석으로 숫자 예측하기

이 노트북은 `book/chapters/ch09_regression_analysis.md`와 `src/regression.py`의 회귀 분석 흐름을 그대로 실행합니다.

핵심은 높은 성능을 만드는 것이 아니라 다음 원칙을 지키는 것입니다.

- 예측 시점을 먼저 정의합니다.
- 주문 상세의 수량·단가·금액은 목표값 생성에만 사용합니다.
- 식별자와 예측 이후 정보를 입력값에서 제외합니다.
- 날짜 순서로 훈련·테스트 데이터를 나눕니다.
- 단순 평균 베이스라인보다 실제로 나은지 확인합니다.
- 낮거나 음수인 R²도 숨기지 않고 해석합니다.


## 0. 실행 전 확인

이 노트북은 5장에서 만든 다음 파일을 사용합니다.

- `data/processed/customers_clean.csv`
- `data/processed/orders_clean.csv`
- `data/processed/order_items_clean.csv`

파일이 없다면 프로젝트 루트에서 다음 명령을 먼저 실행합니다.

```bash
python scripts/preprocess_data.py
```


## 1. 회귀 문제와 예측 시점

예측 대상은 주문별 주문 상세 금액 합계인 `order_total`입니다.

교육용 예측 시점에서는 주문 시점과 결제 수단, 고객의 비식별 특성은 알 수 있지만 주문 상세의 수량·단가·금액은 모델에 제공하지 않는다고 가정합니다.

따라서 `quantity`, `unit_price`, `line_total`, `item_count`, `total_quantity`, `avg_unit_price`, `order_status`, `order_id`, `customer_id`는 입력값으로 사용하지 않습니다.


## 2. 패키지와 프로젝트 경로 설정


In [ ]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

from src.regression import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_leakage_checklist,
    build_regression_dataset,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print("프로젝트 루트:", PROJECT_ROOT.resolve())
print("전처리 데이터:", PROCESSED_DIR.resolve())
print("보고서 폴더:", REPORT_DIR.resolve())


## 3. 전처리 데이터 불러오기

원본 CSV로 자동 대체하지 않습니다. 5장의 전처리 기준과 다른 데이터가 섞이면 실습 결과를 비교하기 어렵기 때문입니다.


In [ ]:
data = load_regression_source_data(PROCESSED_DIR)

customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


## 4. 누수 없는 주문 단위 모델링 데이터 만들기

`order_total`은 주문 상세에서 계산하지만, 주문 상세에서 만든 다른 집계값은 모델 입력에 포함하지 않습니다.


In [ ]:
model_data = build_regression_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

print("모델링 데이터:", model_data.shape)
display(
    model_data[
        [
            "order_id",
            "order_date",
            *FEATURE_COLUMNS,
            TARGET_COLUMN,
        ]
    ].head()
)


In [ ]:
print("숫자형 입력값:", NUMERIC_FEATURES)
print("범주형 입력값:", CATEGORICAL_FEATURES)
print("예측 대상:", TARGET_COLUMN)
print("금지 입력값:", sorted(FORBIDDEN_FEATURES))

validate_feature_columns(FEATURE_COLUMNS)


## 5. 날짜 순서로 훈련·테스트 데이터 분할

과거 주문으로 이후 주문을 예측하는 상황을 모방하기 위해 무작위 분할 대신 날짜 순서 분할을 사용합니다.


In [ ]:
train_data, test_data = split_model_data_by_time(
    model_data,
    test_size=0.2,
)

X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()

print(
    "훈련 기간:",
    train_data["order_date"].min(),
    "~",
    train_data["order_date"].max(),
)
print(
    "테스트 기간:",
    test_data["order_date"].min(),
    "~",
    test_data["order_date"].max(),
)
print("훈련:", X_train.shape, "테스트:", X_test.shape)


## 6. 베이스라인·선형 회귀·랜덤 포레스트 비교

각 모델의 전처리 파이프라인은 훈련 데이터 안에서만 결측치 대체, 표준화, 원-핫 인코딩을 학습합니다.

`Baseline Mean`은 훈련 목표값의 평균을 모든 테스트 주문에 예측합니다. 다른 모델은 최소한 이 기준과 비교해야 합니다.


In [ ]:
models, model_comparison, predictions = (
    train_and_evaluate_models(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        random_state=42,
    )
)

model_comparison


`MAE_improvement_vs_baseline_pct`가 양수이면 베이스라인보다 평균 절대 오차가 줄었습니다. 음수이면 복잡한 모델이 단순 평균보다도 나쁘다는 뜻입니다.

훈련 MAE가 매우 낮고 테스트 MAE가 크면 과적합 가능성을 확인합니다.


## 7. 시간 순서 교차검증

한 번의 테스트 기간 결과만 믿지 않고, 훈련 기간 안에서 여러 시점으로 나누어 성능의 평균과 변동을 확인합니다.


In [ ]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
    max_splits=5,
)

cv_summary


교차검증 MAE 표준편차가 크면 기간에 따라 성능이 불안정할 수 있습니다. R²가 반복적으로 음수라면 현재 입력값으로 평균 예측을 안정적으로 넘지 못할 가능성이 큽니다.


## 8. 실제값·예측값과 잔차 확인

테스트 MAE가 가장 낮은 비베이스라인 모델을 진단용으로 선택합니다. 선택된 모델이 베이스라인보다 낫다는 뜻은 아니므로 비교표를 함께 확인해야 합니다.


In [ ]:
selected_model_name = select_diagnostic_model(
    model_comparison
)

prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)

print("진단 모델:", selected_model_name)
prediction_result.head(10)


In [ ]:
prediction_result["abs_error"].describe()


`residual = 실제값 - 예측값`입니다.

- 잔차가 양수이면 실제값을 낮게 예측했습니다.
- 잔차가 음수이면 실제값을 높게 예측했습니다.
- 큰 절대 오차가 특정 기간이나 금액대에 몰리는지 확인합니다.


## 9. 체크리스트와 결과 저장

예측 결과에는 주문 식별자가 포함되므로 `_internal` 파일은 외부 공개용으로 사용하지 않습니다.


In [ ]:
regression_checklist = build_leakage_checklist()
regression_checklist


In [ ]:
output_paths = save_regression_outputs(
    model_data=model_data,
    train_data=train_data,
    test_data=test_data,
    model_comparison=model_comparison,
    cv_summary=cv_summary,
    prediction_result=prediction_result,
    checklist=regression_checklist,
    selected_model_name=selected_model_name,
    report_dir=REPORT_DIR,
)

for name, path in output_paths.items():
    print(f"{name}: {path}")


저장되는 주요 결과는 다음과 같습니다.

- 모델 비교표
- 시간 순서 교차검증 요약
- 내부용 모델링 데이터와 예측 오차
- 회귀 분석 체크리스트와 Markdown 보고서
- 실제값·예측값 산점도
- 잔차 히스토그램


## 10. 공통 모듈로 전체 파이프라인 다시 실행하기

앞의 단계는 `src/regression.py`의 `run_regression_analysis()`로 한 번에 재현할 수 있습니다.


In [ ]:
regression_result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)

regression_result["model_comparison"]


## 11. 실행 스크립트 사용

프로젝트 루트에서 다음 명령을 실행하면 같은 파이프라인이 수행됩니다.

```bash
python scripts/run_regression_analysis.py
```


## 12. LLM에게 회귀 코드를 요청하는 프롬프트

```text
온라인 쇼핑몰 주문 금액을 예측하는 교육용 회귀 모델을 만들려고 합니다.

예측 대상:
- order_total: 주문별 line_total 합계

예측 시점:
- 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만
  주문 상세의 수량, 단가, 금액은 모델에 제공하지 않음

사용 가능한 입력값:
- payment_method
- order_month
- order_dayofweek
- gender
- age
- city

사용하면 안 되는 입력값:
- order_total, line_total
- quantity, unit_price
- item_count, total_quantity, avg_unit_price
- order_status
- order_id, customer_id

요청:
1. 날짜 순서로 훈련 데이터 80%, 테스트 데이터 20%를 나누어 주세요.
2. 결측치 처리와 OneHotEncoder를 Pipeline 안에 넣어 주세요.
3. DummyRegressor, LinearRegression, RandomForestRegressor를 비교해 주세요.
4. MAE, RMSE, R²와 베이스라인 대비 개선율을 계산해 주세요.
5. 훈련·테스트 성능 차이와 데이터 누수 가능성을 설명해 주세요.
6. R²가 음수일 때의 의미도 설명해 주세요.

주의:
- 실제 데이터에 없는 컬럼을 만들지 마세요.
- 테스트 데이터로 전처리 규칙을 학습하지 마세요.
- 높은 성능을 가정하거나 결과를 임의로 만들어내지 마세요.
```


## 13. 실습 과제

1. `payment_method`를 제외했을 때 베이스라인 대비 성능이 어떻게 달라지는지 비교합니다.
2. `RandomForestRegressor`의 `min_samples_leaf` 값을 바꾸어 훈련·테스트 MAE를 비교합니다.
3. 시간 순서 교차검증에서 성능이 가장 낮은 기간의 데이터 특성을 확인합니다.
4. 예측 시점 이전의 고객 구매 이력을 설계하되 미래 정보가 섞이지 않도록 집계 기준일을 명시합니다.
5. LLM이 제안한 회귀 코드에서 목표값의 계산 재료나 식별자가 입력에 포함됐는지 검토합니다.


In [ ]:
# 과제 1. FEATURE_COLUMNS에서 payment_method를 제외한 별도 실험을 설계해 보세요.


In [ ]:
# 과제 2. RandomForestRegressor의 min_samples_leaf 값을 바꾸어 비교해 보세요.


## 14. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 예측 시점 정의
- 목표값과 목표값 계산 재료 분리
- 식별자와 예측 이후 정보 제외
- 날짜 순서 훈련·테스트 분할
- Pipeline 안에서 결측치 처리와 인코딩
- 평균 베이스라인, 선형 회귀, 랜덤 포레스트 비교
- MAE, RMSE, R²와 베이스라인 대비 개선율 해석
- 시간 순서 교차검증
- 실제값·예측값과 잔차 진단
- 내부 식별 정보 보호
- `src/regression.py`와 실행 스크립트를 통한 재현

낮은 성능도 현재 데이터와 입력 변수의 한계를 알려 주는 유효한 분석 결과입니다.
